## Bath_dMFA_2026 Software 7 — Uncertainty in the future projection

Notebook 6 quantified uncertainty in the *historic* inflow-driven model. Here we
apply the same Monte Carlo ideas to the **future projection** — the stock-driven
model of Notebook 3 — to turn the single best-guess scenario into a fan of
plausible futures out to 2060.

The workflow is the same as before, so **much of the code you need you have
already written in Notebook 6** — we'll prompt you to copy it across and adapt
it to the new model.

This notebook has three parts:

- **Section 1 — prep.** As before, make a neat model function we can call with different parameter values.
- **Section 2 — propagation.** We take the data-quality assessment as read (you saw how it works in Notebook 6) and go straight to deciding *which future scenario inputs are uncertain*, sampling them, and propagating them through the stock-driven model.
- **Section 3 — model uncertainty.** Instead of swapping the whole model (as the leaching example did), we test the effect of the *shape* of the lifetime distribution: a normal vs a Weibull discard curve with the same mean.

### Setup

We load the future scenario data (population, travel demand, occupancy,
kilometrage, the future drive-technology split and the lifetime) and — as the
fixed starting point for the projection — the historic fleet computed in
Notebook 2.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats
from scipy.special import gamma

DATA = "../data/Bath_dMFA_2026_Software_Data.xlsx"
RESULTS2 = "../data/Bath_dMFA_2026_Software_2_Results.xlsx"

# Future scenario drivers (2025-2060, 36 years), best-guess values:
P0 = pd.read_excel(DATA, sheet_name="Population", index_col=0).values[:, 0] * 1e6   # people
pCV0 = pd.read_excel(DATA, sheet_name="Passenger_km", index_col=0).values[:, 0]     # p-km per capita / yr
OR0 = pd.read_excel(DATA, sheet_name="Occupancy rate", index_col=0).values[:, 0]    # persons per car
IU0 = pd.read_excel(DATA, sheet_name="Kilometrage", index_col=0).values[35:, 0]     # km per vehicle / yr (2025-)

# Drive-technology split of the inflow (1990-2060) and mean lifetimes [gasoline, BEV]:
BEVshare_cT = pd.read_excel(DATA, sheet_name="BEV_share", index_col=0).values        # (71, 2), %
life0 = pd.read_excel(DATA, sheet_name="Lifetime", index_col=0).values[0].astype(float)
I_c = pd.read_excel(DATA, sheet_name="New registration", index_col=0).values[:, 0]   # historic inflow

# Fixed historic fleet (1990-2024) from Notebook 2, used to initialise the projection:
HIST_S = np.zeros((71, 71, 2))
HIST_S[0:35, 0:35, 0] = pd.read_excel(RESULTS2, sheet_name="Stock_tc_gasoline", index_col=0).values
HIST_S[0:35, 0:35, 1] = pd.read_excel(RESULTS2, sheet_name="Stock_tc_BEVs", index_col=0).values
HIST_O = np.zeros((71, 71, 2))
HIST_O[0:35, 0:35, 0] = pd.read_excel(RESULTS2, sheet_name="Outflow_tc_gasoline", index_col=0).values
HIST_O[0:35, 0:35, 1] = pd.read_excel(RESULTS2, sheet_name="Outflow_tc_BEVs", index_col=0).values
HIST_IcT = np.zeros((71, 2))
HIST_IcT[0:35] = np.einsum("c,cT->cT", I_c, BEVshare_cT[0:35, :]) / 100

years_full = np.arange(1990, 2061)   # 71 years, for plotting
print("future drivers:", P0.shape, pCV0.shape, OR0.shape, IU0.shape, "| historic fleet:", HIST_S.shape)

### Prep: the stock-driven model as a function

As before, we wrap the model (here the Notebook 3 stock-driven model) into a
single function so we can call it many times.

We define a flexible `lifetime_pdf` function, so we can later test different
distributions/shapes for this. The historic fleet (`HIST_*`) is treated as a
fixed initial condition — the uncertain inputs are the explicit arguments. 

In [ ]:
def lifetime_pdf(lifetime, sigma_frac, pdf_kind):
    """Discard probability by age (shape (2, 71)) for the two technologies, with matched mean."""
    ages = np.arange(71)
    tau = np.asarray(lifetime, float).reshape(2, 1)
    if pdf_kind == "normal":
        return scipy.stats.norm.pdf(ages, tau, sigma_frac * tau)
    elif pdf_kind == "weibull":
        shape = 2.0                                  # a broad, right-skewed survival curve
        scale = tau / gamma(1 + 1 / shape)           # chosen so the mean equals tau
        return scipy.stats.weibull_min.pdf(ages, shape, scale=scale)
    raise ValueError(f"unknown pdf_kind: {pdf_kind}")


def run_stock_driven_model(P_t, pCV_t, OR_t, IU_t, bev_share, lifetime, sigma_frac=0.3, pdf_kind="normal"):
    """Stock-driven projection (Notebook 3) over 1990-2060.

    Parameters (the uncertain inputs)
    ----------
    P_t, pCV_t, OR_t, IU_t : arrays (36,)   future population, p-km/capita, occupancy, kilometrage
    bev_share              : array (71, 2)  gasoline/BEV % split of the inflow
    lifetime               : array (2,)     mean lifetime [gasoline, BEV]
    sigma_frac, pdf_kind   : shape of the discard distribution

    Returns
    -------
    inflow_cT (71, 2), stock_tcT (71, 71, 2), outflow_tcT (71, 71, 2)
    """
    S_future = P_t * pCV_t / (OR_t * IU_t)           # required total fleet size, 2025-2060
    pdf = lifetime_pdf(lifetime, sigma_frac, pdf_kind)

    inflow_cT = HIST_IcT.copy()                      # start from the fixed historic fleet
    stock_tcT = HIST_S.copy()
    outflow_tcT = HIST_O.copy()

    for year in range(35, 71):                       # for all future years (2025-2060)
        stock_tcT[year, :, :] = stock_tcT[year - 1, :, :]   # carry last year's stock forward
        for drivetech in range(2):
            leave = np.zeros(71)
            for age_cohort in range(year + 1):       # age each surviving cohort out
                leave[age_cohort] = inflow_cT[age_cohort, drivetech] * pdf[drivetech, year - age_cohort]
            stock_tcT[year, :, drivetech] -= leave
            outflow_tcT[year, :, drivetech] += leave
        gap = S_future[year - 35] - stock_tcT[year, :, :].sum()   # shortfall vs the demand
        inflow_cT[year, :] = gap * bev_share[year, :] / 100       # new registrations fill the gap
        stock_tcT[year, year, :] = inflow_cT[year, :]             # the new cohort

    return inflow_cT, stock_tcT, outflow_tcT

**Check it reproduces Notebook 3?** e.g. the BEV share of the fleet in 2050?

In [ ]:
inflow_cT, stock_tcT, outflow_tcT = run_stock_driven_model(P0, pCV0, OR0, IU0, BEVshare_cT, life0)

stock_tT = np.einsum("tcT->tT", stock_tcT)
bev_fleet_2050 = 100 * stock_tT[60, 1] / stock_tT[60].sum()
print(f"BEV share of the fleet in 2050: {bev_fleet_2050:.1f}%")

# keep the deterministic ("best guess") trajectories for reference later:
I_det_fut = inflow_cT.sum(1)                 # total new registrations per year
O_det_fut = outflow_tcT.sum(axis=(1, 2))     # total scrappage per year

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(years_full, I_det_fut); ax[0].set_title("New registrations (inflow)")
ax[1].plot(years_full, O_det_fut); ax[1].set_title("End-of-life vehicles (outflow)")
for a in ax:
    a.axvline(2024.5, color="grey", ls=":"); a.set_xlabel("year"); a.set_ylabel("vehicles / yr")
fig.tight_layout();

In [ ]:
CHECKED_MODEL_OUTPUT = False
assert CHECKED_MODEL_OUTPUT, "set to True after you have checked"

## Section 2 — Propagating scenario uncertainty

Now the Monte Carlo. The machinery is identical to Notebook 6 — *sample inputs →
run the model → collect the spread* — so you can reuse most of that code. What
is new is **deciding which future inputs are uncertain**. Unlike the historic
data, the future scenario drivers are really unknown: how much we'll travel
(`pCV`), how full cars will be (`OR`), how far each is driven (`IU`), how long
vehicles last (`lifetime`), how fast the fleet electrifies (`bev_share`), and
the population (`P`). The uncertainty is epistemic, and reflects our belief
about the probability of different future scenarios.

> **Predict first:** for a projection to 2060, which of those inputs do you
think carries the most uncertainty, and which will matter most for the
*scrappage* (outflow)? Write down a guess. 

### Step 1 — a function to draw the uncertain parameters

In Notebook 6 we wrote `draw_sample(rng)`. Write the equivalent here, but **you
decide what to sample and how**. Fill in the body of the function below so it
returns one random set of inputs in the order the model expects: `(P_t, pCV_t,
OR_t, IU_t, bev_share, lifetime)`.

Think about: which drivers are uncertain enough to bother sampling? What spread
(CV) is reasonable for each? Which can you leave at their best-guess value
(`P0`, `pCV0`, `OR0`, `IU0`, `BEVshare_cT`, `life0`)? Keep any
physically-bounded quantity sensible, as you did for the BEV share and lifetime
in Notebook 6. 

In [ ]:
def draw_future_parameters(rng):
    """Return one random (P_t, pCV_t, OR_t, IU_t, bev_share, lifetime), in that order.
    You decide which inputs are uncertain and by how much; leave the rest at best guess."""
    ...

**Check it:** call it a few times and confirm the inputs you chose to sample
really vary (and the others don't). For example, the mean lifetimes: 

In [ ]:
for _ in range(5):
    P0, pCV, OR, IU, BEVshare_cT, lifetime = draw_future_parameters(rng=np.random.default_rng())
    print(lifetime)   # or other parameter

### Step 2 — the Monte Carlo loop

This is the same loop you wrote in Notebook 6 (Section 2.3) — copy it across and
adapt it: the arrays now span 71 years (1990–2060), the model is
`run_stock_driven_model`, the draw is `draw_future_parameters`. Save the *total
outflow* per year (across all age cohorts and vehicle types) to `O_fut_samples`.

In [ ]:
rng = np.random.default_rng(20260701)

In [ ]:
N = 500
O_fut_samples = ...   # an (N, 71) array; row i = total outflow per year for sample i
for i in range(N):
    ...

As before, check that the `O_fut_samples` are behaving as expected. For example,
check the shape is as expected, look at the first few samples for the most
recent years; and check that the mean across the samples should is similar to
the deterministic line from the Prep section. 

### Step 3 — the fan chart to 2060

Again, copy your fan-chart code from Notebook 6 (Section 2.7) and adapt it to
`O_fut_samples` and `years_full`, with the deterministic `O_det_fut` line on
top. A vertical line at 2024.5 would be nice to separate the fixed history from
the widening projection (see `plt.axvline`). 

In [ ]:
...

You should see that the spread is zero through the (fixed) history and fans out
into the future, reaching its widest by 2060.

> **Exercise**: Try re-running with more or fewer uncertain input parameters
sampled in `draw_future_parameters`. What seems to make the biggest difference
to the spread of the results? 

## Section 4 — Model uncertainty: the shape of the lifetime distribution

We previously tested the effect of the assumed model structure by swapping in a
completely different (leaching) model. Here instead we'll keep the same model
structure but change the *shape* of the discard distribution. So far we assumed
a *normal* curve; an alternative is the *Weibull* distribution. With the *same
mean lifetime*, do the two shapes give materially different projections? 

In [ ]:
ages = np.arange(71)
pdf_n = lifetime_pdf(life0, 0.3, "normal")
pdf_w = lifetime_pdf(life0, 0.3, "weibull")

plt.figure(figsize=(8, 4))
plt.plot(ages, pdf_n[0], label="normal")
plt.plot(ages, pdf_w[0], label="Weibull")
plt.title(f"Discard probability by age — gasoline (mean lifetime {life0[0]:.0f} yr)")
plt.xlabel("age of vehicle (years)"); plt.ylabel("probability of discard"); plt.xlim(0, 40); plt.legend();

The Weibull here is broader than the normal: it allows some earlier scrappage
*and* a longer tail of old survivors, at the same mean lifetime. Now we'll
propagate that structural choice through to the result.

**Your turn:** run the projection (on the best-guess inputs) with
`pdf_kind="weibull"`, and plot its total outflow against the normal-distribution
result `O_det_fut`, on top of the Section 2–3 scenario fan. Does the
distribution-shape difference sit inside the scenario-uncertainty band, or
outside it? 

In [ ]:
...

Unlike the comparison of the lifetime vs leaching stock model in the previous
Notebook 6, the shape of the lifetime distribution is relatively unimportant.
This is consistent to the findings of [Miatto et al
(2017)](https://doi.org/10.1016/j.resconrec.2017.01.015).

> **Optional exercise:** This result is now inconsistent, as it assumes a normal
    lifetime distribution curve for the historical analysis, and then a Weibull
    distribution for the future model. Try going back and repeating the
    historical analysis with a Weibull curve; how much difference does it make
    there? Is there a more dramatic difference if you adjust the shape parameter
    of the Weibull curve? 

### Conclusions

We've taken the same uncertainty analysis we did in Notebook 6, and applied it
to a forward projection.

The code and analysis is essentially the same, whether we are dealing with
historical uncertainty or uncertainty about future scenarios. However, the way
you define the ranges of possible parameter values may be quite different.
Depending on your research questions, you may want to model future uncertainty
as parameter distributions, like here — or consider discrete scenarios without
giving the scenarios a specific probability.